# 🎯 [K09_M04] 답변이 자료와 맞는지 확인하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K09/k09_m04_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·상담 (Level 2) / K09 부대 정보 검색 시스템 |
| **미션** | 미션 04. 답변이 자료와 맞는지 확인하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 프롬프트 엔지니어링 및 팩트체크 검증 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k09_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K09", "../uploads/K09"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k09_m01~m10) 준비 완료!")



---

## 🎯 미션 04. 답변이 자료와 맞는지 확인하기
**소요시간**: 약 1.5시간

![미션 4 원문 대조 검증 및 출처 표기 의무화 다이어그램](/uploads/common/K09_m04_factcheck_citation.png)

### 📋 미션 개요 및 배경
생성형 AI가 안내 자료를 잘못 해석하거나 누락·왜곡하여 발생하는 오안내 위험을 통제하기 위해 답변 대조 검증을 수행합니다. 생성된 답변의 운영 시간·장소·연락처를 원본 안내 자료와 1:1로 비교하여 '맞음/틀림/지어냄'으로 분류하고, 답변 끝에 반드시 근거 시설명을 표기하도록 프롬프트를 개선합니다. AI 결과를 맹신하지 않고 인간이 직접 검증하는 HITL(Human-in-the-Loop) 실무 습관을 확립합니다.

### 🎯 미션 학습목표
AI 답변의 시간·위치·연락처를 자료표와 1:1로 대조하여 맞음·틀림·지어냄으로 정량 판정하고, 답변 끝에 출처 근거(시설명)를 의무 표기하도록 지시문을 개선합니다. **이를 통해 AI의 환각(Hallucination) 현상을 식별·검증하는 팩트체크 기법과 출처 표기를 유도하는 프롬프트 통제 기술을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 답변을 대조표에 맞음·틀림·지어냄으로 표시한다.
#### 이 단계의 역할
미션3에서 수집된 AI 답변 문장을 단위 사실(시간, 장소, 대상, 조건)로 분해하여 원문 자료표와 1:1 대조 판정합니다.

#### 해야 할 일

<li>답변 대조표 서식을 준비하고 질문 5건의 AI 답변 문장을 단위 정보별로 분해합니다.

</li><li>원문 자료표와 대조하여 일치하는 정보는 '맞음', 수치 오차는 '틀림', 원문에 없는 창작은 '지어냄'으로 라벨링합니다.

</li><li>오류가 발생한 항목의 원인(프롬프트 지시 모호성, 수치 오독 등)을 정밀 분석합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[단위 사실 교차 검증 질의]:** "다음 [AI 생성 답변]을 아래 [부대 안내 자료]와 한 문장씩 비교 검증해 줘. 각 문장이 사실인지(맞음), 숫자가 다른지(틀림), 자료에 아예 없는 내용인지(지어냄) 판정하고 근거를 제시해 줘.
[부대 안내 자료]: {미션2 자료표}
[AI 생성 답변]: {미션3 생성 답변}"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **엄밀한 육안 대조:** 시간(분 단위), 요일, 장소 명칭이 글자 그대로 일치하는지 육안으로 재확인했는가?
- **판정의 객관성:** 환각(지어냄) 요소가 엄격하게 감지되었는가?

#### 핵심 확인 지표
<blockquote>답변 문장별 맞음/틀림/지어냄 판정 대조표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>답변 대조표(필수 산출물)

</li>

#### 🔹 Step 2. 틀린 답의 질문 표현을 기록한다.
#### 이 단계의 역할
어떤 유형의 질문 표현(모호한 질문, 복합 질문, 군 축약어)이 AI의 오답이나 왜곡을 유발했는지 상관관계를 도출합니다.

#### 해야 할 일

<li>대조표에서 '틀림' 또는 '지어냄'으로 판정된 질문들의 표현 방식을 분석합니다.

</li><li>두 가지를 한 번에 물어본 경우(복합 질문)나 시설 명칭을 줄여 부른 경우 등 오류 유발 패턴을 기록합니다.

</li><li>장병들이 질문을 작성할 때 유의해야 할 사용자 가이드라인 초안을 정리합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[오류 유발 질문 패턴 분석 질의]:** "장병들이 입력한 다음 질문 표현들 중 AI가 잘못된 답변을 생성하게 만든 표현상 취약점(모호성, 복합 질의, 약어 등)을 분석하고, 이를 방지할 수 있는 명확한 질문 변환 예시를 작성해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **오류 원인 규명:** 단순 오답 표기가 아닌 질문 구조적 원인이 규명되었는가?
- **개선 방향 도출:** 질문 표현을 어떻게 고쳐야 정답률이 올라가는지 해결책이 제시되었는가?

#### 핵심 확인 지표
<blockquote>질문 표현별 오류 발생 원인 분석표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>틀린 답의 질문 표현 분석 기록지

</li>

#### 🔹 Step 3. 근거 시설명을 쓰게 한다.
#### 이 단계의 역할
답변의 신뢰도를 높이고 검증을 용이하게 하기 위해 답변 끝에 반드시 인용한 '근거 항목(시설명)'을 명기하도록 지시문을 개선합니다.

#### 해야 할 일

<li>프롬프트 지시사항에 "답변 말미에 [근거: OOO 안내] 형식으로 출처 시설명을 표기하라"는 규칙을 추가합니다.

</li><li>개선된 프롬프트로 5개 질문을 재실행하여 출처 표기가 올바르게 출력되는지 확인합니다.

</li><li>표기된 근거 시설명이 실제 원문의 해당 섹션과 정확히 일치하는지 대조합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[출처 표기 의무화 프롬프트 개선]:** "[개선 지시사항 추가]: ...반드시 답변의 마지막 줄에 [근거 시설: 시설명] 형태로 정보를 인용한 원본 시설 명칭을 명시해야 함. 출처를 특정할 수 없는 내용은 답변에 포함하지 말 것.
[질문]: PX 이용 시간 알려줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **출처 표기 준수율:** 모든 답변 끝에 지정된 형식([근거 시설: ...])이 누락 없이 출력되었는가?
- **출처 일치성:** 답변 내용과 표기된 근거 시설명이 실제로 부합하는가?

#### 핵심 확인 지표
<blockquote>근거 시설명 표기 의무화 프롬프트

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>근거 표기 프롬프트(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
근거 시설명뿐만 아니라 해당 자료의 행 번호(Row ID)까지 더욱 정밀하게 인용하도록 프롬프트를 고도화합니다.

#### 해야 할 일
답변 끝에 근거뿐만 아니라 해당 시설의 대표 업무 1가지를 덧붙이게 해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[행 번호 정밀 인용 질의]:** "답변 작성 시 반드시 [근거: No.03 의무실 진료 안내]처럼 자료표의 행 번호와 시설명을 함께 표기하도록 프롬프트를 수정해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **정밀 매핑:** 자료표 행 번호와 시설명이 오차 없이 정확히 매핑되었는가?

#### 핵심 확인 지표
<blockquote>정밀 출처 표기 프롬프트 및 검증 시간 단축 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 최적화 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M04] 답변이 자료와 맞는지 확인하기 실습 가이드 및 작성 서식

## 1. AI 답변-원문 1:1 교차 대조표 (Fact-checking Sheet)

| 문항 | AI 답변 문장 분해 | 원본 자료표 대조 내용 | 판정 (맞음/틀림/지어냄) | 오류 원인 및 보완 사항 |
| :--- | :--- | :--- | :---: | :--- |
| **Q1-1** | "의무실은 본관 1층 서편에 위치합니다." | 본관 1층 서편 일치 | **맞음** | 사실 일치 |
| **Q1-2** | "평일 진료는 09:00~17:30까지입니다." | 평일 09:00~17:30 일치 | **맞음** | 수치 일치 |
| **Q1-3** | "12:00~13:30은 소독 시간입니다." | 12:00~13:30 소독 일치 | **맞음** | 조건 일치 |
| **Q2-1** | "충성마트는 평일 12:00~20:00까지입니다." | 원문은 19:30 마감 | **틀림** | AI가 20:00로 숫자 왜곡 |
| **Q2-2** | "주말에는 언제든 자유롭게 방문 가능합니다."| 원문: 일요일 휴무 | **지어냄** | 일요일 휴무 조건 누락 및 창작 |

---

## 2. 근거(출처) 표기 의무화 프롬프트 템플릿

```markdown
[지시사항 수정/추가]:
- 반드시 답변의 마지막 줄에 [근거 시설: OO 안내] 형식으로 참조한 자료표의 정확한 시설 명칭을 명기할 것.
- 인용한 출처를 특정할 수 없는 내용은 답변에 일체 포함하지 말 것.

[출력 예시]:
답변: 대대 의무실 일반 진료는 평일 09:00부터 17:30까지입니다. (12:00~13:30 소독 제외)
[근거 시설: 대대 의무실 운영 안내]
```

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **단위 사실 교차 검증 대조표 양식 (Fact-checking Sheet):** 시간, 위치, 연락처 정보별 '맞음/틀림/지어냄' 1:1 판정 서식
- **출처(근거) 표기 의무화 프롬프트 템플릿:** 답변 말미에 [근거: 시설명]을 강제 출력하도록 설계된 개선 프롬프트
- **[학습 가이드] AI 환각(Hallucination) 감지 및 원문 대조 검증(HITL) 수칙:** AI의 왜곡·날조 패턴 유형별 식별 요령 및 장병 직접 검증 습관 가이드

**실습 팁:** AI 답변에서 분 단위 숫자(예: 17:30 vs 18:00)나 요일 조건은 대충 넘어가기 쉬우므로, 원문 글자를 하나씩 손가락으로 짚어가며 육안 검증하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] AI 결과 검증 — AI 답변 문장을 원문과 1:1 대조하여 오류를 식별하고 출처 근거를 표기하도록 프롬프트를 개선했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>답변 문장별 맞음/틀림/지어냄 1:1 대조 점검표 완성

</li><li>오류 유발 질문 표현 패턴 분석 완료

</li><li>답변 끝 근거 시설명 표기 프롬프트 개선 완료

</li>

#### 채점자 판정 메모
<blockquote>단순히 맞다고 넘어가지 않고, 분 단위 시간이나 요일 조건의 미세한 왜곡을 짚어내었는지 대조표의 판정 근거를 확인한다.

</blockquote>





In [ ]:
# [미션 04. 답변이 자료와 맞는지 확인하기] 모범 프롬프트 및 완성 결과 기록
mission_04_prompt = """
[단위 사실 1:1 교차 검증 질의 (Fact-checking)]
다음 [AI 생성 답변]을 아래 [부대 안내 자료표]의 원문과 문장 단위로 대조 검증(Fact-checking)하라.
각 문장별로 [맞음 / 틀림 / 지어냄(환각)]을 판정하고, 판정 근거와 대조 원본 내용을 작성하라.
또한 향후 모든 답변에 인용 근거(자료 출처 및 시설명)를 의무적으로 표기하도록 프롬프트 지시사항을 보완하라.

[AI 생성 답변]:
"의무실은 본관 1층 서편에 위치합니다. 평일 진료 시간은 09:00~17:30이며 주말에도 오전 10시까지 진료합니다. 점심 소독 시간은 12:00~13:30이며 일반 진료가 중단됩니다. 문의는 내선 103번입니다."
"""

mission_04_result = """
# [K09_M04] 답변이 자료와 맞는지 확인하기 실습 가이드 및 작성 서식

## 1. AI 답변-원문 1:1 교차 대조표 (Fact-checking Sheet)

| 문항 | AI 답변 문장 분해 | 원본 자료표 대조 내용 | 판정 (맞음/틀림/지어냄) | 오류 원인 및 보완 사항 |
| :--- | :--- | :--- | :---: | :--- |
| **Q1-1** | "의무실은 본관 1층 서편에 위치합니다." | 본관 1층 서편 일치 | **맞음** | 사실 일치 |
| **Q1-2** | "평일 진료는 09:00~17:30까지입니다." | 평일 09:00~17:30 일치 | **맞음** | 수치 일치 |
| **Q1-3** | "12:00~13:30은 소독 시간입니다." | 12:00~13:30 소독 일치 | **맞음** | 조건 일치 |
| **Q2-1** | "충성마트는 평일 12:00~20:00까지입니다." | 원문은 19:30 마감 | **틀림** | AI가 20:00로 숫자 왜곡 |
| **Q2-2** | "주말에는 언제든 자유롭게 방문 가능합니다."| 원문: 일요일 휴무 | **지어냄** | 일요일 휴무 조건 누락 및 창작 |

---

## 2. 근거(출처) 표기 의무화 프롬프트 템플릿

```markdown
[지시사항 수정/추가]:
- 반드시 답변의 마지막 줄에 [근거 시설: OO 안내] 형식으로 참조한 자료표의 정확한 시설 명칭을 명기할 것.
- 인용한 출처를 특정할 수 없는 내용은 답변에 일체 포함하지 말 것.

[출력 예시]:
답변: 대대 의무실 일반 진료는 평일 09:00부터 17:30까지입니다. (12:00~13:30 소독 제외)
[근거 시설: 대대 의무실 운영 안내]
```
"""

# 렌더러 실행 (모범 실습 결과를 시각적으로 확인합니다)
show_result("미션 04. 답변이 자료와 맞는지 확인하기", mission_04_prompt, mission_04_result)

